# Abstract graph operators: overview

This notebook is a broad, hands-on tour of decomposition operators. Each example builds a decomposition function, applies it to a graph, and displays the resulting abstract graph and the mappings back to the original graph. The later sections show how the same decompositions support vectorization and feature inspection.


In [ ]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


The next cell loads the notebook bootstrap and common scientific Python tools used by the examples. The bootstrap makes the package available whether this notebook is opened from the ecosystem repository or from the `abstractgraph` repository itself.


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import numpy as np
import scipy as sp
import pandas as pd
import networkx as nx
import random
import matplotlib.pyplot as plt
# import warnings filter
from warnings import simplefilter
# ignore all future warnings
simplefilter(action='ignore', category=FutureWarning)
from IPython.core.display import HTML
HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell;text-align: center;vertical-align: middle;}</style>')

These imports provide the graph display helpers and the operator constructors. The `draw` helper below runs a decomposition on a graph, then shows both the operator pipeline and the abstract graph it produces.


In [ ]:
from abstractgraph.graphs import AbstractGraph, graph_to_abstract_graph
from abstractgraph.display import display, display_graph, display_mappings, display_decomposition_graph, decomposition_to_graph
from abstractgraph.labels import graph_hash_label_function_factory
from abstractgraph.operators import *

def draw(graph, df, nbits=11):
    display_decomposition_graph(df)
    ag = graph_to_abstract_graph(graph, decomposition_function=df, nbits=nbits)
    display(ag, size=(12,6))
    display_mappings(ag, n_elements_per_row=10)

Choose the input graph for the examples. The default builds a molecule graph from a SMILES string; set `USE_EXAMPLE` to `'random'` to use a generated graph instead. The display helps connect later results to the original structure.


In [ ]:
USE_EXAMPLE = 'chemical' # 'random' or 'chemical'

if USE_EXAMPLE == 'random':
    from abstractgraph import RandomGraphConstructor
    graph = RandomGraphConstructor(integers_range=12, instance_size=40, alphabet_size=4, attribute_size=3).sample(1)
else:
    smi = 'CC1NC(OC2CC(O)C3(CO)C4C(O)CC5(C)C(C6CNC(=O)C6)CCC5(O)C4CCC3(O)C2)C(O)C(O)C1O'
    from abstractgraph_graphicalizer.chem import MoleculeGraphicalizer, draw_molecule
    graph = MoleculeGraphicalizer().transform([smi])[0]
    draw_molecule(graph)

print(graph)

display_graph(graph)

Start with the simplest decompositions: `node()` selects node-based pieces, while `intersection_edges()` keeps track of how those pieces overlap along input edges. This shows how a basic operator can be wrapped to capture connections between its results.


In [ ]:
df = compose(intersection_edges(), node())
draw(graph, df)

Here the same edge-intersection wrapper is applied to `edge()`, so the decomposition is built from individual edges and their overlaps. Compare it with the node-based version above.


In [ ]:
df = compose(intersection_edges(), edge())
draw(graph, df)

`random_part` samples a fixed number of parts from the graph. Wrapping it in `intersection_edges()` also records connections between the sampled pieces.


In [ ]:
df = compose(intersection_edges(), random_part(n_samples=6))
draw(graph, df)

`add` combines the results of two decompositions, here cycles and trees. The outer `intersection_edges()` then adds overlap information across the combined result.


In [ ]:
df = compose(intersection_edges(), add(cycle(), tree()))
draw(graph, df)

`path` extracts path-shaped subgraphs whose edge counts fall between two and three. The range lets one operator describe several related path sizes.


In [ ]:
df = path(number_of_edges=(2,3))
draw(graph, df)

`union_of_shortest_paths` collects shortest paths subject to a length range. This is useful when the desired subgraphs are defined by routes between graph elements.


In [ ]:
df = union_of_shortest_paths(length=(3,5))
draw(graph, df)

A radius-one `neighborhood` creates local views around graph elements, including the radius-zero case. Varying the radius is a compact way to control how much nearby structure each result contains.


In [ ]:
df = neighborhood(radius=(0,1))
draw(graph, df)

`unlabel()` removes labels from the neighborhoods before returning them. This lets structurally matching neighborhoods be treated alike even when their original labels differ.


In [ ]:
df = compose(neighborhood(radius=(0,1)), unlabel())
draw(graph, df)

This example adds labeled and unlabelled neighborhood decompositions together. It demonstrates that `add` can preserve complementary views of the same graph in one decomposition.


In [ ]:
df1 = compose(neighborhood(radius=(0,1)), unlabel())
df2 = neighborhood(radius=(0,1))
df = add(df1, df2)
draw(graph, df) 

The pipeline builds trees of several sizes, labels each size group, and combines them with cycles. `unlabel()` and `restore_label()` control when labels are ignored during matching and restored in the final result; `intersection_edges()` records connections between the resulting pieces.


In [ ]:
df2 = compose(prepend_label(label='2'), filter_by_number_of_nodes(number_of_nodes=2), tree())
df3 = compose(prepend_label(label='3'), filter_by_number_of_nodes(number_of_nodes=3), tree())
df4 = compose(prepend_label(label='4+'), filter_by_number_of_nodes(number_of_nodes=(4,100)), tree())
df1 = add(cycle(), df2, df3, df4)
df = compose(intersection_edges(), restore_label(), df1, unlabel())
draw(graph, df)

This creates one node decomposition for each distance from one to three, tags each result with that distance, and combines them. The labels make the distance setting visible in the output.


In [ ]:
df_list = [compose(combination(distance=d), node(), prepend_label(label=d)) for d in range(1,4)]
df = add(*df_list)
draw(graph, df)

`combination` selects small groups of elements at distance zero, then `cycle()` finds cyclic structure within each group. The results are unlabelled so structurally equivalent cycles can be grouped together.


In [ ]:
df = compose(combination(number_of_elements=(2,3), distance=0), cycle(), unlabel())
draw(graph, df)

`graphlet` extracts small local graph patterns around each element, constrained here to three or four nodes within radius one.


In [ ]:
df = graphlet(radius=1, number_of_nodes=(3,4))
draw(graph, df)

`betweenness_centrality` selects subgraphs around highly central structure, limited here to four nodes. This is an example of a decomposition driven by a graph measure rather than only by local neighborhoods.


In [ ]:
df = betweenness_centrality(number_of_nodes=4)
draw(graph, df)

This builds central cores using a split form of betweenness centrality, then derives connector pieces from the edges outside those cores. The final union removes redundant mapped subgraphs and adds edge-intersection information.


In [ ]:
core_df = compose(connected_component(), betweenness_centrality_split(number_of_nodes=9))
connector_df = compose(connected_component(), edge_complement(), merge(use_edges=True), core_df)
df = compose(intersection_edges(), remove_redundant_mapped_subgraphs(), add(connector_df, core_df))
draw(graph, df)

The `intersection` operator restricts neighborhood results to matches with sizes between three and five. This illustrates using an intersection constraint to limit another decomposition.


In [ ]:
df = compose(intersection(node_size=(3,5)), neighborhood(radius=2))
draw(graph, df)

These pipelines compare central subgraphs with central subgraphs found in their complements. Combining the core, peripheral, and connector results gives a decomposition that describes both selected structure and the surrounding context.


In [ ]:
from abstractgraph.operators import *
n = 3
core_df = compose(connected_component(), betweenness_centrality(number_of_nodes=n))
peripheral_df = compose(connected_component(), complement(), betweenness_centrality(number_of_nodes=n))
centrality_df = add(peripheral_df,core_df)

merged_centrality_df = add(
        compose(complement(), betweenness_centrality(number_of_nodes=n)),
        betweenness_centrality(number_of_nodes=n)
    )

connector_df = compose(connected_component(), edge_complement(), merge(use_edges=True), merged_centrality_df)
df = compose(intersection_edges(), remove_redundant_mapped_subgraphs(), add(connector_df, centrality_df))
draw(graph, df)

The graph is split into connected components, then nodes are grouped by degree ranges from one through four or more. The final intersection combines these groups while disabling edge-based connection acceptance.


In [ ]:
df1 = compose(connected_component(), degree(value=(1,1)))
df2 = compose(connected_component(), degree(value=(2,2)))
df3 = compose(connected_component(), degree(value=(3,3)))
df4 = compose(connected_component(), degree(value=(4,10)))
df = compose(intersection_edges(accept_connection_by_edge=False), add(df1,df2,df3,df4))
draw(graph, df)

`merge()` first brings graph pieces together; the following operators keep connected components and select those with nodes of degree one or two. This shows how a merge can precede structural filtering.


In [ ]:
df = compose(merge(), connected_component(), degree(value=(1,2)))
draw(graph, df)

The complement is computed before cycle extraction. This finds cycles in the structure left out by the original graph pieces.


In [ ]:
df = compose(complement(), cycle())
draw(graph, df)

This filters graph elements by their labels, requiring label `0` and excluding label `2`, then generates neighborhoods from the survivors.


In [ ]:
df = compose(filter_by_node_label(must_have_one_of=[0], cannot_have_any_in=[2]), neighborhood())
draw(graph, df)

This keeps neighborhoods whose abstract subgraphs contain two or three nodes. The size filter is applied before the neighborhood operation in the composed pipeline.


In [ ]:
df = compose(filter_by_number_of_nodes(number_of_nodes=(2,3)), neighborhood())
draw(graph, df)

This applies the analogous constraint to edge counts, retaining neighborhoods with two or three edges.


In [ ]:
df = compose(filter_by_number_of_edges(number_of_edges=(2,3)), neighborhood())
draw(graph, df)

The pipeline combines edges at distances one or two, then keeps results with exactly two connected components.


In [ ]:
df = compose(filter_by_number_of_connected_components(number_of_components=(2,2)), combination(distance=(1,2)), edge())
draw(graph, df)

This repeats the same distance-based edge combinations but keeps connected results with exactly one component, making the connectedness constraint easy to compare with the previous example.


In [ ]:
df = compose(filter_by_number_of_connected_components(number_of_components=(1,1)), combination(distance=(1,2)), edge())
draw(graph, df)

Here cycles and trees are combined, then edge intersections connect their mapped results. The next cell changes the intersection size threshold to show how that setting affects the connections.


In [ ]:
df = compose(intersection_edges(), add(cycle(),tree()))
draw(graph, df)

`size_threshold=2` requires a larger shared edge intersection than the default. Compare the output with the previous cell to see which connections are filtered out.


In [ ]:
df = compose(intersection_edges(size_threshold=2), add(cycle(),tree()))
draw(graph, df)

`clique` extracts complete subgraphs with one to four nodes. The size range includes single-node cliques as well as larger fully connected groups.


In [ ]:
df = clique(number_of_nodes=(1,4))
draw(graph, df)

This combines small groups of elements at distances one or two, then expands each group to radius-zero or radius-one neighborhoods.


In [ ]:
df = compose(combination(number_of_elements=(1,2),distance=(1,2)), neighborhood(radius=(0,1)))
draw(graph, df)

The helper packages a neighborhood-based NSPPK-style decomposition. Its `radius` controls local context, `distance` controls how far apart selected elements can be, and `thickness` chooses whether to include shortest-path unions as well.


In [ ]:
#NSPPK
def make_nsppk(radius=1, distance=3, thickness=1):
    if thickness == 0:
        df = add(union_of_shortest_paths(length=distance), compose(combination(number_of_elements=2,distance=(0,distance)), neighborhood(radius=(0,radius))))
    elif thickness == 1:
        df = compose(combination(number_of_elements=2,distance=(0,distance)), neighborhood(radius=(0,radius)))
    else:
        raise Exception(f'thickness {thickness} is not allowed')
    return df

df = make_nsppk(radius=1, distance=3, thickness=1)
draw(graph, df)

This selects combinations with two or three connected components and expands them to neighborhoods of radius up to two. It demonstrates combining a structural constraint with a local expansion.


In [ ]:
df = compose(
    filter_by_number_of_connected_components(number_of_components=(2,3)), 
    combination(number_of_elements=(2,3),distance=(1,2)), neighborhood(radius=(0,2)))
draw(graph, df)

`compose_product` evaluates `cycle()` and `tree()` on the same input graph, then sends both result sets to `binary_combination`. This differs from ordinary sequential composition, where one operator's output feeds into the next.


In [ ]:
#Note: compose_product(binary_combination(distance=(0,1)), cycle(), tree()) computes cycle(Q) and tree(Q) on the same input and feeds both to binary_combination.
df = compose_product(binary_combination(distance=0), cycle(), tree())
draw(graph, df)

---

# Branching and bounded iteration

The next examples use conditional and repeated execution to build decompositions whose steps depend on intermediate results.


# Control flow and iterative decomposition

Operators can also branch or repeat based on properties of intermediate abstract graphs. These examples use predicates over the current result to decide which transformations to apply.


In [ ]:
step1 = compose(connected_component(), complement(), betweenness_centrality(number_of_nodes=5))

@curry
def number_of_interpretation_graph_nodes_greater_then(abstract_graph, threshold=1):
    return number_of_interpretation_graph_nodes(abstract_graph) > threshold

workflow = compose(
    if_then_else(
        predicate=number_of_interpretation_graph_nodes_greater_then(threshold=2),
        then_function=merge(),
        else_function=identity()
    ),
    step1,
)
draw(graph, workflow)


`for_loop` applies the neighborhood decomposition three times. `deduplicate()` then removes repeated results that may arise as the neighborhoods expand.


In [ ]:
df1 = neighborhood()
df = for_loop(function=df1, n_iterations=3)
df = compose(deduplicate(), df)
draw(graph, df)

`split` partitions the input into five pieces. The pipeline also derives connector pieces from their edge complement, then combines cores and connectors while recording edge intersections.


In [ ]:
from abstractgraph.operators import *
core_df = split(n_parts=5)
connector_df = compose(connected_component(), edge_complement(), merge(use_edges=True), core_df)
df = compose(intersection_edges(), add(connector_df, core_df))

draw(graph, df)

This uses a `while_loop` to keep splitting while the largest current subgraph has at least five nodes, with a maximum of ten iterations. The resulting cores are then joined with connector pieces derived from their complements.


In [ ]:
@curry
def number_of_interpretation_graph_nodes_less_then(abstract_graph, threshold=1):
    return number_of_interpretation_graph_nodes(abstract_graph) < threshold

@curry
def number_of_subgraph_nodes_greater_then(abstract_graph, threshold=1):
    return max_number_of_subgraph_nodes(abstract_graph) >= threshold


core_df = forward_compose(
    while_loop(
        function=split(),
        predicate=number_of_subgraph_nodes_greater_then(threshold=5),
        max_iterations=10
    ),
    connected_component()
)

connector_df = compose(connected_component(), edge_complement(), merge(use_edges=True), core_df)
df = compose(intersection_edges(), add(connector_df, core_df))

draw(graph, df)

This combines three candidate decompositions—radius-one neighborhoods, unlabelled cycles, and connected combinations of cycles. Such a union can supply multiple candidate views for later feasibility filters.


In [ ]:
#decomposition used for feasibility filters
df1 = neighborhood(radius=1)
df2 = compose(cycle(), unlabel())
df3 = compose(filter_by_number_of_connected_components(number_of_components=1), combination(distance=0), compose(cycle(), unlabel()))
df = add(df1,df2,df3)
draw(graph, df)

---

# Feasibility filters and paired results

These examples build candidate subgraphs under structural constraints, then combine candidates from separate pipelines.


# Combining operators and serializing a pipeline

The first example forms pairs from two separately filtered cycle decompositions: one constrained by size and one by the presence of an `N` label. `compose_product` passes both sets to a binary combination operator, and edge intersections connect the resulting pairs.


In [ ]:
cycle_df = compose(filter_by_number_of_nodes(number_of_nodes=(5,6)), cycle())
cycle_with_N_df = compose(filter_by_node_label(must_have_one_of=['N']), cycle())
pairs_df = compose_product(binary_combination(distance=(1,4)), cycle_df, cycle_with_N_df)
df = compose(intersection_edges(), pairs_df)
draw(graph, df)

Operator pipelines can be saved as XML and reconstructed later. This cell registers the available operators, prints an XML representation of the current decomposition, then deserializes it back into `df_rt`. The final display currently draws `df`, so use `df_rt` there if you want to inspect the reconstructed pipeline directly.


In [ ]:
# Example ofthe XML rendering 

from abstractgraph.xml import register_from_module, operator_to_xml_string, operator_from_xml_string
import abstractgraph.operators as ag_ops
register_from_module(ag_ops)
xml_df = operator_to_xml_string(df, pretty=True)
print(xml_df)

df_rt = operator_from_xml_string(xml_df)
draw(graph, df)


---


# Vectorization

The operator outputs can be converted into fixed-width numeric features for machine-learning workflows. The examples below cover one graph, graph batches, and node-level matrices.


A decomposition can be converted into a numeric feature representation. The first example applies `node()` and `edge()` to one random graph, then `vectorize` returns a fixed-width feature vector; `nbits` sets the size of the hashed feature space.


In [ ]:
from abstractgraph.graphs import AbstractGraph
from abstractgraph.vectorize import vectorize
from abstractgraph.operators import *
decomposition_function = add(node(), edge())
from abstractgraph import RandomGraphConstructor
graph = RandomGraphConstructor(integers_range=12, instance_size=40, alphabet_size=4, attribute_size=3).sample()

ag = AbstractGraph(graph=graph).create_default_interpretation_node()
ag = decomposition_function(ag)
X = vectorize(ag, nbits=10)
X.shape

`AbstractGraphTransformer` applies one decomposition to a collection of graphs and returns a matrix with one feature row per graph. Here each graph is represented by its radius-zero to radius-two neighborhoods.


In [ ]:
from abstractgraph.vectorize import AbstractGraphTransformer
from abstractgraph.operators import *
from abstractgraph import RandomGraphConstructor
graphs = RandomGraphConstructor(integers_range=12, instance_size=40, alphabet_size=4, attribute_size=3).sample(1000)
df = neighborhood(radius=(0,2))
X = AbstractGraphTransformer(decomposition_function=df, nbits=10, n_jobs=-1).fit_transform(graphs)
X.shape

`AbstractGraphNodeTransformer` returns node-level feature matrices, one per input graph. The list length corresponds to the number of graphs, while each matrix has rows for nodes and columns for hashed features.


In [ ]:
from abstractgraph.vectorize import AbstractGraphNodeTransformer
from abstractgraph import RandomGraphConstructor
graphs = RandomGraphConstructor(integers_range=12, instance_size=40, alphabet_size=4, attribute_size=3).sample(1000)
df = neighborhood(radius=(0,2))
X_list = AbstractGraphNodeTransformer(decomposition_function=df, nbits=10, n_jobs=-1).fit_transform(graphs)
len(X_list), X_list[0].shape

---

# Feature inspection

Hashed feature labels can be traced back to the subgraphs that produced them. The examples below inspect individual labels and summarize collisions.


# Inspecting hashed features

Hashed vector columns can be mapped back to representative subgraphs. This example uses a small graph sample and a deliberately compact hash space (`nbits=3`) so collisions are easy to inspect.


In [ ]:
from abstractgraph import RandomGraphConstructor
graphs = RandomGraphConstructor(integers_range=12, instance_size=40, alphabet_size=4, attribute_size=3).sample(5)

from abstractgraph.operators import *
df = neighborhood(radius=2)

from abstractgraph.feature_subgraphs import display_feature_subgraphs
display_feature_subgraphs(
    graphs=graphs,
    decomposition_function=df,
    nbits=3,
)

This repeats feature-to-subgraph inspection over a larger graph sample and several hash widths. The histograms show how many subgraphs map to each feature label, illustrating how the hash width changes collisions.


In [ ]:

from abstractgraph import RandomGraphConstructor
graphs = RandomGraphConstructor(integers_range=12, instance_size=40, alphabet_size=4, attribute_size=3).sample(100)

from abstractgraph.operators import *
df = neighborhood(radius=2)

from abstractgraph.feature_subgraphs import feature_subgraphs

for nbits in range(6,12):
    label_map = feature_subgraphs(
        graphs=graphs,
        decomposition_function=df,
        nbits=nbits,
    )
    lens = [len(label_map[k]) for k in label_map]
    plt.figure(figsize=(7,3))
    plt.hist(lens,range(1,max(lens)))
    plt.xticks(range(1,max(lens)))
    plt.title(f'nbits:{nbits}')
    plt.xlabel('#collisions')
    plt.show()

# Meta analysis

The following cells inspect the decomposition pipeline itself as a graph. This makes operator order and data flow visible, in addition to the abstract graph produced when the pipeline runs.


# Meta analysis

The final examples examine the decomposition function as an operator graph, showing the pipeline's structure separately from its output on an input graph.


This constructs a pipeline with a bounded `while_loop`, then converts that pipeline into a graph and displays its nodes and edges. The printed attributes expose each operator and its connections for closer inspection.


In [ ]:
df = forward_compose(
    while_loop(
        function=split(),
        predicate=number_of_interpretation_graph_nodes_less_then(threshold=5),
        max_iterations=10
    ),
    intersection_edges(accept_connection_by_edge=True)
)

display_decomposition_graph(decomposition_to_graph(df))

G = decomposition_to_graph(df)
nx.draw(G)

print('-'*100)
print('Nodes')
for u in G.nodes():
    print(u,G.nodes[u])
print('-'*100)
print('Edges')
for u,v in G.edges():
    print('%s -> %s [%s]'%(u,v,G.edges[u,v]))

The final cell reapplies the cycle-and-tree union with edge intersections to an undirected NetworkX view of the graph. It serves as a compact closing example of composing multiple operators.


In [ ]:
df = compose(intersection_edges(), add(cycle(),tree()))
draw(nx.Graph(G), df)

---
